# Step-by-step implementation
The following code demonstrates how to integrate step-back prompting into an RAG pipeline. Let’s break it down step-by-step:
1. Import necessary libraries
2. Set up the OpenAI API key
3. Few-shot learning for step-back prompting
4. Build the step-back prompt
5. Retrieve information
6. Build the RAG chain

## 1. Import necessary libraries

In [26]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.runnables import RunnableLambda
from helpers import get_experientiallabs_llm
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_classic import hub

## 2. Set up the LangSmith

In [27]:
# get_experientiallabs_llm() reads EXPERIENTIALLABS_API_KEY, so no OpenAI key is needed here.

In [28]:
os.environ['LANGSMITH_PROJECT']='Step-Back_Prompting'

## 3. Few-shot learning for step-back prompting

In [29]:
# Few Shot Examples
examples = [
    {
        "input": "Did the Beatles ever write a book?",
        "output": "What types of creative works did the Beatles produce?"
    },
    {
        "input": "Was Albert Einstein a musician?",
        "output": "What fields did Albert Einstein work in?"
    }
]

# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

## 4. Build the step-back prompt

In [30]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        few_shot_prompt,
        ("user", "{question}"),
    ]
)

In [31]:
question_gen = prompt | get_experientiallabs_llm() | StrOutputParser()

In [32]:
question = "Did Leonardo da Vinci invent the printing press?"

In [33]:
question_gen.invoke({"question": question})

'What inventions and innovations are attributed to Leonardo da Vinci?'

## 5. Retrieve information

In [34]:
search = DuckDuckGoSearchAPIWrapper(max_results=4)

def retriever(query):
    return search.run(query)

In [35]:
retriever(question)

'6 Feb 2026 · Leonardo da Vinci C) Leonardo da Vinci B: printing press with movable type had already been invented by Johannes Guttenberg around the year 1450 and it spread ... 6 Feb 2026 · The First Printing Press was Invented by Johannes Gutenberg in 1440. Johannes Gutenberg, a German blacksmith, goldsmith, printer, and inventor, is credited with ... Johannes Gutenberg and his printing press - Facebook DaVinci: Joked about how he must have invented everything ... More results from www.facebook.com 13 Feb 2026 · Invented the printing press? Answer, A, Johanna Schutenberg. Leonardo da Vinci Quiz 17 Nov 2025 · Printing multiplied information. The printing press accelerated the reproduction of texts, images, maps, religious arguments, and technical knowledge.'

In [36]:
retriever(question_gen.invoke({"question": question}))

"28 Aug 2026 · Leonardo da Vinci, the Renaissance intellect, revolutionized art and science with such masterpieces as the Mona Lisa while pioneering advancements in ... 25 Oct 2025 · Try Opera yourself: https://opr.as/Opera-browser-lostintimevids I have made 3d animations of Leonardo da Vinci's most impressive inventions, and collected ... 22 Sept 2025 · Leonardo da Vinci is remembered today not only for masterpieces like the Mona Lisa and The Last Supper, but also for his notebooks filled with mechanical ... 17 Nov 2025 · Leonardo presented himself as someone able to design bridges, siege equipment, fortifications, artillery-related devices, and machines for both attack and ..."

## 6. Build the RAG chain

In [37]:
from langsmith import Client

response_prompt = Client().pull_prompt("langchain-ai/stepback-answer", dangerously_pull_public_prompt=True)

In [38]:
chain = (
    {
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        "step_back_context": question_gen | retriever,
        "question": lambda x: x["question"],
    }
    | response_prompt
    | get_experientiallabs_llm()
    | StrOutputParser()
)

In [39]:
chain.invoke({"question": question})

'No. Leonardo da Vinci did **not** invent the printing press. Johannes Gutenberg, a German goldsmith and printer, developed the first practical movable-type printing press in Europe around **1440–1450**, before Leonardo was born in 1452.\n\nLeonardo did, however, study and redesign aspects of printing technology. His notebooks include ideas for improving presses and making printing more efficient, but these were modifications and inventions related to printing—not the invention of the printing press itself.'